In [ ]:
#Install necessary packages
%pip install azure-ai-projects==2.0.0b2 openai==1.109.1 python-dotenv azure-identity


In [ ]:
#Setting up environment variables
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition

import jsonref

load_dotenv()

foundry_project_endpoint =os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")


In [ ]:
#Setting up the project client
project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential(),
)

In [ ]:
with open("weather_openapi.json", "r") as f:
    openapi_weather = jsonref.loads(f.read())

#Intialize agent OpenAPI tool using the read in OpenAPI spec

weather_tool={
    "type":"openapi",
    "openapi":{
        "name":"weather",
        "spec":openapi_weather,
        "auth":{
            "type":"anonymous"
        },
    }
}

In [ ]:
#Creating Agent with the weather tool
agent_name = "weather-agent"

agent = project_client.agents.create_version(
    agent_name=agent_name,
    definition=PromptAgentDefinition(   
        model=model_deployment_name,
        instructions="You are a helpful weather assistant. Use the provided OpenAPI tool to answer questions about the weather.",
        tools=[
            weather_tool
        ]
    ),
)
print(f"Agent created with name: {agent.id} {agent.name} and version: {agent.version}")

In [ ]:
openai_client = project_client.get_openai_client()

conversation = openai_client.conversations.create()
print(f"Created conversation with id: {conversation.id}")

In [ ]:
user_input = "What's the weather like in Seattle today?"

In [ ]:
response = openai_client.responses.create(
            conversation=conversation.id,
            extra_body={
                "agent":{
                    "name": agent_name,
                    "type": "agent_reference",
                }
            },
            
            input=user_input,
    )

print(f"Agent: {response.output_text}")